In [2]:
# %pip install --force-reinstall --upgrade Pillow

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
from torchvision import datasets, transforms

In [4]:
# 1. 하이퍼파라미터 설정하기
batch_size = 64
learning_rate = 0.001
epochs = 100

# 장치가 gps가 있으면 gpu쓰고 없으면 cpu 쓰기
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# 2. 데이터 준비
# 이미지 전처리를 위한 변환 정의
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, ), (0.5, ))
])

In [6]:
# 3. 데이터셋 다운로드 및 로드
train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = datasets.MNIST(root="./data", train=False, transform=transform, download=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)

100%|██████████| 9.91M/9.91M [00:02<00:00, 3.75MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 166kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 1.83MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.4kB/s]


In [13]:
# 4. 완전연결신경망 만들기
class ImageClassifier(nn.Module):
    def __init__(self, input_size=28*28, hidden_size=500, num_class=10):
        # 그림 크기가 28 by 28
        super(ImageClassifier, self).__init__() # 첫번째 인자 클래스명, 두번쨰 인자:객체자신
        self.fc1 = nn.Linear(input_size, hidden_size) # 784 -> 500
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_class) # 500 -> 10

    def forward(self, x):
        # 이미지를 1차원 벡터로 만들어서 전달해야 한다.
        x = x.reshape(-1, 28*28)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [14]:
# 5. 모델 만들기
model = ImageClassifier()

In [15]:
# 6. 손실함수와 옵티마이저 정의하기
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [23]:
def train_model(epochs=100):
    for epoch in range(epochs):
        for i, (images, labels) in enumerate(train_loader):
            # 순전파
            outputs = model(images)
            loss = criterion(outputs, labels) # 손실계산
            optimizer.zero_grad() # 가중치 초기화
            loss.backward() # 역전파
            optimizer.step() # 가중치 업데이트
            if (i + 1) % 100 == 0:
                print(f"Epochs [{epoch + 1} / {epochs}] Step [{i + 1} / {len(train_loader)} Loss {loss.item():.4f}]")             


In [24]:
def evaluate_model():
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels in test_loader:
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)        
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    print(f"테스트셋 정확도 {accuracy:.2f}%")

In [25]:
if __name__ == "__main__":
    train_model()
    evaluate_model()

Epochs [1 / 100] Step [100 / 938 Loss 0.2323]
Epochs [1 / 100] Step [200 / 938 Loss 0.3258]
Epochs [1 / 100] Step [300 / 938 Loss 0.1599]
Epochs [1 / 100] Step [400 / 938 Loss 0.1558]
Epochs [1 / 100] Step [500 / 938 Loss 0.1246]
Epochs [1 / 100] Step [600 / 938 Loss 0.2384]
Epochs [1 / 100] Step [700 / 938 Loss 0.1870]
Epochs [1 / 100] Step [800 / 938 Loss 0.1036]
Epochs [1 / 100] Step [900 / 938 Loss 0.3345]
Epochs [2 / 100] Step [100 / 938 Loss 0.2077]
Epochs [2 / 100] Step [200 / 938 Loss 0.2556]
Epochs [2 / 100] Step [300 / 938 Loss 0.1482]
Epochs [2 / 100] Step [400 / 938 Loss 0.0936]
Epochs [2 / 100] Step [500 / 938 Loss 0.1316]
Epochs [2 / 100] Step [600 / 938 Loss 0.1016]
Epochs [2 / 100] Step [700 / 938 Loss 0.0892]
Epochs [2 / 100] Step [800 / 938 Loss 0.0475]
Epochs [2 / 100] Step [900 / 938 Loss 0.1508]
Epochs [3 / 100] Step [100 / 938 Loss 0.1096]
Epochs [3 / 100] Step [200 / 938 Loss 0.1447]
Epochs [3 / 100] Step [300 / 938 Loss 0.0909]
Epochs [3 / 100] Step [400 / 938 L